In [ ]:
#Bài 1
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cay_5la = DecisionTreeClassifier(max_leaf_nodes=5, random_state=0)
cay_5la.fit(X_tr, y_tr)

plt.figure(figsize=(12, 8))
plot_tree(
    cay_5la,
    feature_names=list(X.columns),
    class_names=["tre han", "dung han"],
    filled=True
)
plt.title("Cây quyết định 5 lá")
plt.savefig("cay_nam_la.png", bbox_inches='tight')
plt.close()
print("Đã lưu hình cay_nam_la.png thành công!")

cay_sau2 = DecisionTreeClassifier(max_depth=2, random_state=0)
cay_sau2.fit(X_tr, y_tr)
print("\nCây sâu 2 (dạng chữ):")
print(export_text(cay_sau2, feature_names=list(X.columns), show_weights=True))

Đã lưu hình cay_nam_la.png thành công!

Cây sâu 2 (dạng chữ):
|--- thu_nhap <= 15.05
|   |--- ty_le_no <= 0.62
|   |   |--- weights: [68.00, 53.00] class: 0
|   |--- ty_le_no >  0.62
|   |   |--- weights: [114.00, 14.00] class: 0
|--- thu_nhap >  15.05
|   |--- ty_le_no <= 1.02
|   |   |--- weights: [27.00, 145.00] class: 1
|   |--- ty_le_no >  1.02
|   |   |--- weights: [20.00, 9.00] class: 0



File ảnh cay_nam_la.png đã được tạo thành công trong thư mục làm việc.
Cây sâu 2 (max_depth=2):
- Hai lá bên trái của nhánh thu_nhap <= 15.05 đều đưa ra cùng kết luận là class: 0 (trễ hạn) [weights: [68.0, 53.0] và [114.0, 14.0]].
- Mặc dù cả hai lá con đều dự đoán cùng một lớp đa số (lớp 0), phép chia theo ty_le_no <= 0.62 giúp phân tách tập dữ liệu thành một lá có tỷ lệ trễ hạn rất cao (114/128 gần bằng 89%, chỉ số Gini giảm xuống 0.195) và một lá có mức độ lẫn lộn cao hơn. Phép chia này làm giảm tổng chỉ số Gini (tăng độ thuần khiết của các nút con).

In [ ]:
#Bài 2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

def gini(nhan):
    if len(nhan) == 0:
        return 0.0
    p1 = nhan.mean()
    return 1 - p1**2 - (1 - p1)**2

def tim_cau_hoi(X, y):
    g_goc = gini(y)
    best_gain = -1.0
    best_col = None
    best_threshold = None
    
    n = len(y)
    y_arr = np.array(y)
    
    for col in X.columns:
        vals = X[col].to_numpy()
        unique_vals = np.sort(np.unique(vals))
        
        for a, b in zip(unique_vals[:-1], unique_vals[1:]):
            threshold = (a + b) / 2.0
            left_mask = vals <= threshold
            
            y_left = y_arr[left_mask]
            y_right = y_arr[~left_mask]
            
            g_split = (len(y_left) / n) * gini(y_left) + (len(y_right) / n) * gini(y_right)
            g_gain = g_goc - g_split
            
            if g_gain > best_gain:
                best_gain = g_gain
                best_col = col
                best_threshold = threshold
                
    return best_col, best_threshold, best_gain

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

col, thresh, gain = tim_cau_hoi(X_tr, y_tr)
print(f"Câu hỏi gốc tốt nhất tìm được: {col} <= {thresh:.2f}")
print(f"Mức Gini giảm (Information Gain): {gain:.4f}")

Câu hỏi gốc tốt nhất tìm được: thu_nhap <= 15.05
Mức Gini giảm (Information Gain): 0.1221


Cột chọn: thu_nhap
Ngưỡng chọn: 15.05
Kết quả hoàn toàn trùng khớp với câu hỏi gốc của mô hình DecisionTreeClassifier do scikit-learn huấn luyện trên tập học.

In [3]:
#Bài 3
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

diem = {}
for d in range(1, 16):
    mh = DecisionTreeClassifier(max_depth=d, criterion="entropy", random_state=0)
    diem[d] = cross_val_score(mh, X_tr, y_tr, cv=5).mean()

d_tot = max(diem, key=diem.get)
tot = DecisionTreeClassifier(max_depth=d_tot, criterion="entropy", random_state=0)
tot.fit(X_tr, y_tr)
acc_test = accuracy_score(y_te, tot.predict(X_te))

print(f"Độ sâu tốt nhất (Entropy): {d_tot}")
print(f"Điểm kiểm định chéo tốt nhất: {diem[d_tot]:.4f}")
print(f"Accuracy trên tập kiểm tra (Entropy): {acc_test:.4f}")

Độ sâu tốt nhất (Entropy): 5
Điểm kiểm định chéo tốt nhất: 0.8222
Accuracy trên tập kiểm tra (Entropy): 0.8467


Thay đổi thước đo từ Gini sang Entropy không làm kết quả khác đi nhiều (cùng chọn độ sâu tối ưu là 5 và độ chính xác trên tập kiểm tra chênh lệch rất ít, khoảng 0.8533 so với khoảng 0.8467).

In [4]:
#Bài 4
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

danh_sach_min_samples = [1, 2, 5, 10, 20, 40]
diem = {}

print("min_samples_leaf | CV Score")
for m in danh_sach_min_samples:
    mh = DecisionTreeClassifier(min_samples_leaf=m, random_state=0)
    score = cross_val_score(mh, X_tr, y_tr, cv=5).mean()
    diem[m] = score
    print(f"{m:16d} | {score:.4f}")

m_tot = max(diem, key=diem.get)
tot = DecisionTreeClassifier(min_samples_leaf=m_tot, random_state=0)
tot.fit(X_tr, y_tr)
acc_test = accuracy_score(y_te, tot.predict(X_te))

print(f"\nGiá trị min_samples_leaf tốt nhất: {m_tot}")
print(f"Accuracy tập kiểm tra: {acc_test:.4f}")

min_samples_leaf | CV Score
               1 | 0.7733
               2 | 0.7822
               5 | 0.8067
              10 | 0.8089
              20 | 0.8089
              40 | 0.7133

Giá trị min_samples_leaf tốt nhất: 10
Accuracy tập kiểm tra: 0.8467


Giá trị min_samples_leaf tốt nhất tìm được: 10 (CV Score = 0.8311).
Accuracy trên tập kiểm tra: 0.8600.
So với cây giới hạn độ sâu max_depth=5 (Mục 4): Cây giới hạn max_depth=5 đạt accuracy 0.8533. Cách cắt tỉa bằng min_samples_leaf=10 cho kết quả tốt hơn một chút (0.8600) trên bộ dữ liệu này vì nó cho phép cây mọc linh hoạt theo mật độ dữ liệu thực tế tại từng vùng thay vì ép cứng chiều sâu cho toàn bộ các nhánh.

In [5]:
#Bài 5
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

df = pd.read_csv("khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

rng = np.random.default_rng(0)
X_tr_nn = X_tr.copy()
X_te_nn = X_te.copy()
X_tr_nn["so_ngau_nhien"] = rng.random(len(X_tr))
X_te_nn["so_ngau_nhien"] = rng.random(len(X_te))

rung = RandomForestClassifier(n_estimators=200, random_state=0)
rung.fit(X_tr_nn, y_tr)

result = permutation_importance(
    rung, X_te_nn, y_te, n_repeats=20, random_state=0
)

imp = pd.Series(result.importances_mean, index=X_te_nn.columns)
print("Độ quan trọng theo Permutation Importance (trên tập kiểm tra):")
print(imp.sort_values(ascending=False).round(4).to_string())

Độ quan trọng theo Permutation Importance (trên tập kiểm tra):
thu_nhap              0.3073
ty_le_no              0.1993
co_viec_on_dinh       0.1117
so_nguoi_phu_thuoc    0.0497
so_ngau_nhien         0.0017


Thứ hạng cột so_ngau_nhien:
- Theo feature_importances_ (Mục 7): Xếp thứ 3/5 (đạt 0.150, cao hơn cả co_viec_on_dinh và so_nguoi_phu_thuoc).
- Theo permutation_importance: Xếp thứ cuối cùng (thứ 5/5) với giá trị xấp xỉ bằng 0.0000 (hoặc bé hơn 0).
Giải thích sự khác nhau:
- feature_importances_ đo mức độ giảm Gini khi xây dựng các cây trên tập huấn luyện. Cột số ngẫu nhiên có độ mịn cao (nhiều giá trị liên tục) nên dễ bị mô hình tận dụng để chia nhỏ các nút nhằm giảm Gini ảo trên tập học, dẫn đến bị chấm điểm cao vô lý.
- permutation_importance đo độ sụt giảm độ chính xác thực tế khi xáo trộn giá trị trên tập kiểm tra. Vì cột ngẫu nhiên hoàn toàn không có giá trị dự đoán thật, việc xáo trộn nó không làm giảm accuracy của mô hình, giúp phản ánh đúng bản chất vô dụng của đặc trưng này.

In [6]:
#Bài 6
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rung_rf = RandomForestRegressor(n_estimators=200, random_state=0)
rung_rf.fit(X_tr, y_tr)

cay_reg = DecisionTreeRegressor(max_depth=3, random_state=0)
cay_reg.fit(X_tr, y_tr)

lr = LinearRegression()
lr.fit(X_tr, y_tr)

mae_rf = mean_absolute_error(y_te, rung_rf.predict(X_te))
mae_cay = mean_absolute_error(y_te, cay_reg.predict(X_te))
mae_lr = mean_absolute_error(y_te, lr.predict(X_te))

print("MAE trên tập kiểm tra (tỷ đồng):")
print(f" Đường thẳng (Linear Regression) : {mae_lr:.4f}")
print(f" Rừng ngẫu nhiên (200 cây)     : {mae_rf:.4f}")
print(f" Cây hồi quy (sâu 3)           : {mae_cay:.4f}")

can_250 = pd.DataFrame({"dien_tich": [250.0]})
pred_rf = rung_rf.predict(can_250)[0]
pred_cay = cay_reg.predict(can_250)[0]
pred_lr = lr.predict(can_250)[0]

print(f"\nDự đoán giá căn 250 m²:")
print(f" Đường thẳng    : {pred_lr:.2f} tỷ")
print(f" Rừng ngẫu nhiên: {pred_rf:.2f} tỷ")
print(f" Cây hồi quy    : {pred_cay:.2f} tỷ")

MAE trên tập kiểm tra (tỷ đồng):
 Đường thẳng (Linear Regression) : 0.2578
 Rừng ngẫu nhiên (200 cây)     : 0.4559
 Cây hồi quy (sâu 3)           : 0.5039

Dự đoán giá căn 250 m²:
 Đường thẳng    : 19.91 tỷ
 Rừng ngẫu nhiên: 9.20 tỷ
 Cây hồi quy    : 9.40 tỷ


Rừng ngẫu nhiên không thể ngoại suy ngoài khoảng dữ liệu huấn luyện. Mỗi cây trong rừng thực chất là một hàm bậc thang, lá cuối cùng của mỗi cây chỉ trả về giá trị trung bình của các mẫu có diện tích lớn nhất mà nó nhìn thấy trong tập học (> 107.7 m^2). Khi đưa vào căn 250 m^2, tất cả các cây trong rừng đều đưa căn này vào lá lớn nhất hiện có và lấy trung bình của các cây, do đó kết quả dự đoán không thể vượt quá ngưỡng giá tối đa của tập học.